In [1]:
import polars as pl

In [ ]:
df = pl.scan_csv("../auth.txt.gz",has_header=False,separator=",",
                 new_columns= ['time','src_user','dest_user','src_comp','dest_comp',
                               'auth_type','logon_type','auth_orientation','outcome'])

In [7]:
# Count number of rows 
df.select(pl.len()).collect().item()

1051430459

In [ ]:
# Converting seconds
seconds_per_hour = 60 * 60 
seconds_per_day = 60 * 60 * 24

In [ ]:
# Day 1 of 58
subset = df.filter(pl.col('time') < seconds_per_day)

In [ ]:
# Number of rows in day 1
subset.select(pl.len()).collect().item()

15740768

In [ ]:
# Features dataframe
user_features = (
subset
    .with_columns(
        hour=(pl.col("time") // seconds_per_hour).cast(pl.Int64),
        is_fail=(pl.col("outcome") == "Fail").cast(pl.Int64),
    )
    .group_by(["src_user", "hour"])
    .agg(
        n_auths=pl.len(),
        n_fails=pl.col("is_fail").sum(),
    )
    .with_columns(
        failure_ratio=pl.col("n_fails") / pl.col("n_auths"),
    )
    .collect()
)

In [27]:
user_features.show()

src_user,hour,n_auths,n_fails,failure_ratio
str,i64,u32,i64,f64
"""C1409$@DOM1""",4,45,0,0.0
"""U6915@DOM1""",22,4,0,0.0
"""C2510$@DOM1""",4,60,0,0.0
"""NETWORK SERVICE@C6108""",16,1,0,0.0
"""C4429$@DOM1""",13,16,0,0.0


In [30]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
X = user_features.select(["n_auths", "failure_ratio"]).to_numpy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)